# Feature Engineering

Creating user features, movie features, and merging data for recommendation models.

## Overview

- Load processed data from `01_data_exploration.ipynb`
- Engineer user features (engagement, bias, diversity)
- Engineer movie features (popularity, quality, genres)
- Merge all features
- Create train/test split
- Save processed data

## Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import os
import pickle

from sklearn.model_selection import train_test_split


print("Libraries imported successfully!")

Libraries imported successfully!


## Load Data

In [2]:
ratings = pd.read_csv(
    'D:/Study/MLDS/recommendation-engine/data/raw/ml-1m/ratings.dat',
    sep='::',
    engine='python',
    names=['UserID', 'MovieID', 'Rating', 'Timestamp'],
    header=None
)

movies = pd.read_csv(
    'D:/Study/MLDS/recommendation-engine/data/raw/ml-1m/movies.dat',
    sep='::',
    engine= 'python',
    names=['MovieID', 'Title', 'Genres'],
    header=None,
    encoding='latin-1'
)

users = pd.read_csv(
    'D:/Study/MLDS/recommendation-engine/data/raw/ml-1m/users.dat',
    sep='::',
    engine='python',
    names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'],
    header=None
)

print("Data loaded successfully!")
print(f"Ratings shape: {ratings.shape}")
print(f"Movies shape: {movies.shape}")
print(f"Users shape: {users.shape}")

Data loaded successfully!
Ratings shape: (1000209, 4)
Movies shape: (3883, 3)
Users shape: (6040, 5)


## User Features

Creating features that describe each user's behavior:

1. **total_ratings**: How many movies did the user rate?
2. **avg_rating_given**: What's their average rating? (1-5)
3. **rating_std**: How diverse are their ratings? (low = consistent, high = varied)
4. **user_bias**: How much higher/lower do they rate vs global average?

In [3]:
global_avg_rating = ratings['Rating'].mean()
print(f"Global average rating: {global_avg_rating:.3f}")

Global average rating: 3.582


In [4]:
user_features = ratings.groupby('UserID').agg({
    'MovieID': 'count',
    'Rating': ['mean', 'std']
}).reset_index()

In [5]:
user_features.columns = ['UserID', 'total_ratings', 'avg_rating_given', 'rating_std']

In [6]:
user_features['rating_std'] = user_features['rating_std'].fillna(0)
user_features['user_bias']  = user_features['avg_rating_given'] - global_avg_rating

In [7]:
print("\nUser Features Created!")
print(user_features.head(10))
print(f"\nShape: {user_features.shape}")
print("\nStatistics:")
print(user_features.describe())


User Features Created!
   UserID  total_ratings  avg_rating_given  rating_std  user_bias
0       1             53          4.188679    0.680967   0.607115
1       2            129          3.713178    1.001513   0.131614
2       3             51          3.901961    0.984985   0.320396
3       4             21          4.190476    1.077917   0.608912
4       5            198          3.146465    1.132699  -0.435100
5       6             71          3.901408    0.830747   0.319844
6       7             31          4.322581    0.747757   0.741016
7       8            139          3.884892    0.925321   0.303328
8       9            106          3.735849    0.820010   0.154285
9      10            401          4.114713    0.837740   0.533149

Shape: (6040, 5)

Statistics:
            UserID  total_ratings  avg_rating_given   rating_std    user_bias
count  6040.000000    6040.000000       6040.000000  6040.000000  6040.000000
mean   3020.500000     165.597517          3.702705     1.01018

## Movie Features

Creating features that describe each movie:

1. **total_ratings**: How many people rated this movie?
2. **avg_rating_received**: What's the average rating for this movie? (1-5)
3. **movie_bias**: How much above/below global average is this movie?
4. **genres_encoded**: One-hot encode genres (Drama=1/0, Comedy=1/0, etc.)

In [8]:
# Calculate movie features
movie_features = ratings.groupby('MovieID').agg({
    'UserID': 'count',               #total_ratings
    'Rating': ['mean', 'std']        #avg_ratings_received, rating_std

}).reset_index()

#Flatten column names
movie_features.columns = ['MovieID', 'total_ratings', 'avg_ratings_received', 'rating_std']

#Handle NaN values
movie_features['rating_std'] = movie_features['rating_std'].fillna(0)

#calulate movie bias
movie_features['movie_bias'] = movie_features['avg_ratings_received'] - global_avg_rating


#Merge with movie titles and genres
movie_features = movie_features.merge(movies, on='MovieID', how='left')

#Result
print("\nMovie Features Created!")
print(movie_features.head(10))
print(f"\nShape: {movie_features.shape}")
print("\nStatistics:")
print(movie_features.describe())


Movie Features Created!
   MovieID  total_ratings  avg_ratings_received  rating_std  movie_bias  \
0        1           2077              4.146846    0.852349    0.565282   
1        2            701              3.201141    0.983172   -0.380423   
2        3            478              3.016736    1.071712   -0.564828   
3        4            170              2.729412    1.013381   -0.852153   
4        5            296              3.006757    1.025086   -0.574808   
5        6            940              3.878723    0.934588    0.297159   
6        7            458              3.410480    0.979918   -0.171084   
7        8             68              3.014706    0.954059   -0.566859   
8        9            102              2.656863    1.048290   -0.924702   
9       10            888              3.540541    0.891233   -0.041024   

                                Title                        Genres  
0                    Toy Story (1995)   Animation|Children's|Comedy  
1        

## One-Hot Encoding Movie Genres

Converting categorical genre data into numerical features for the hybrid recommendation model.

### Why One-Hot Encoding Here?

**For Content-Based Filtering:**
- Our hybrid model needs to match movies by similarity
- Example: If user liked "Toy Story" (Animation|Children's|Comedy), recommend similar movies
- To calculate similarity, we need numerical representation of genres
- One-hot encoding allows us to compute cosine similarity between genre vectors

**Why not other approaches?**
- Fixed set of 18 genres (perfect for one-hot encoding)
- Simple, interpretable features for production
- Fast inference at serving time
- No need for embeddings or NLP models (overkill for categorical data)

### Trade-off:

One-hot encoding creates 18 new columns (one per genre), but:
- ✅ Fast to compute similarity
- ✅ Easy to interpret (1 = movie has this genre)
- ✅ Scalable (18 genres is manageable)
- ✅ Production-ready

### Result:

Transforms genre text into numerical vectors that can be:
1. Compared with user preference vectors
2. Used in cosine similarity calculations
3. Combined with collaborative filtering scores in the hybrid model

In [9]:
#One-hot encode genres
genres_encoded = movie_features['Genres'].str.get_dummies(sep='|')

#Concatenate with movie_features
movie_features_final = pd.concat(
    [movie_features[['MovieID', 'total_ratings', 'avg_ratings_received', 'rating_std', 'movie_bias']],
     genres_encoded],
     axis=1
)

#Results
print("\nMovie Features with One-Hot Encoded Genres Created!")
print(f"Shape: {movie_features_final.shape}")
print(f"\nGenres Columns: {list(genres_encoded.columns)}")
print(f"Total Genres: {len(genres_encoded.columns)}")
print("\nFirst 10 movies with Genres One-Hot Encoded:")
print(movie_features_final.head(10))



Movie Features with One-Hot Encoded Genres Created!
Shape: (3706, 23)

Genres Columns: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
Total Genres: 18

First 10 movies with Genres One-Hot Encoded:
   MovieID  total_ratings  avg_ratings_received  rating_std  movie_bias  \
0        1           2077              4.146846    0.852349    0.565282   
1        2            701              3.201141    0.983172   -0.380423   
2        3            478              3.016736    1.071712   -0.564828   
3        4            170              2.729412    1.013381   -0.852153   
4        5            296              3.006757    1.025086   -0.574808   
5        6            940              3.878723    0.934588    0.297159   
6        7            458              3.410480    0.979918   -0.171084   
7        8             68              3.014706 

## Merging All Features

Combining user features, ratings, and movie features into one dataset.

Each row will represent: (UserID, MovieID, Rating, user_features, movie_features)

This is the final dataset we'll use for training the recommendation models.

In [10]:
# Merge user features with ratings
ratings_with_features = ratings.merge(user_features, on='UserID', how='left')

# Merge with movie features
ratings_with_features = ratings_with_features.merge(movie_features_final, on='MovieID', how='left')

# Select relevant columns (remove Title since we don't need it for modeling)
# Keep: UserID, MovieID, Rating (target), user features, movie features
columns_to_keep = ['UserID', 'MovieID', 'Rating'] + \
                  [col for col in user_features.columns if col != 'UserID'] + \
                  [col for col in movie_features_final.columns if col != 'MovieID']

In [11]:
# Check column names
print("user_features columns:")
print(user_features.columns.tolist())
print("\nmovie_features_final columns:")
print(movie_features_final.columns.tolist())
print("\nratings_with_features columns:")
print(ratings_with_features.columns.tolist())

user_features columns:
['UserID', 'total_ratings', 'avg_rating_given', 'rating_std', 'user_bias']

movie_features_final columns:
['MovieID', 'total_ratings', 'avg_ratings_received', 'rating_std', 'movie_bias', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

ratings_with_features columns:
['UserID', 'MovieID', 'Rating', 'Timestamp', 'total_ratings_x', 'avg_rating_given', 'rating_std_x', 'user_bias', 'total_ratings_y', 'avg_ratings_received', 'rating_std_y', 'movie_bias', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [12]:
# Fix column name typos and conflicts
ratings_with_features = ratings_with_features.rename(columns={
    'total_ratings': 'user_total_ratings',           # Fix typo
    'avg_rating_given': 'user_avg_rating',
    'rating_std_x': 'user_rating_std',              # User's std
    'user_bias': 'user_bias',
    'total_ratings': 'movie_total_ratings',         # Make explicit
    'avg_ratings_received': 'movie_avg_rating',
    'rating_std_y': 'movie_rating_std',             # Movie's std
    'movie_bias': 'movie_bias'
})


In [14]:
# Just check what columns we have after rename
print("Columns after rename:")
print(ratings_with_features.columns.tolist())

Columns after rename:
['UserID', 'MovieID', 'Rating', 'Timestamp', 'total_ratings_x', 'user_avg_rating', 'user_rating_std', 'user_bias', 'total_ratings_y', 'movie_avg_rating', 'movie_rating_std', 'movie_bias', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [15]:
# Rename the remaining columns properly
ratings_with_features = ratings_with_features.rename(columns={
    'total_ratings_x': 'user_total_ratings',
    'total_ratings_y': 'movie_total_ratings'
})

# Now select the final columns
columns_to_keep = ['UserID', 'MovieID', 'Rating', 
                   'user_total_ratings', 'user_avg_rating', 'user_rating_std', 'user_bias',
                   'movie_total_ratings', 'movie_avg_rating', 'movie_rating_std', 'movie_bias',
                   'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 
                   'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 
                   'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

final_data = ratings_with_features[columns_to_keep]

print("All Features Merged Successfully!")
print(f"Shape: {final_data.shape}")
print(f"\nColumns ({len(final_data.columns)} total):")
print(final_data.columns.tolist())
print(f"\nFirst 5 rows:")
print(final_data.head())
print(f"\nData Info:")
print(final_data.info())

All Features Merged Successfully!
Shape: (1000209, 29)

Columns (29 total):
['UserID', 'MovieID', 'Rating', 'user_total_ratings', 'user_avg_rating', 'user_rating_std', 'user_bias', 'movie_total_ratings', 'movie_avg_rating', 'movie_rating_std', 'movie_bias', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

First 5 rows:
   UserID  MovieID  Rating  user_total_ratings  user_avg_rating  \
0       1     1193       5                  53         4.188679   
1       1      661       3                  53         4.188679   
2       1      914       3                  53         4.188679   
3       1     3408       4                  53         4.188679   
4       1     2355       5                  53         4.188679   

   user_rating_std  user_bias  movie_total_ratings  movie_avg_rating  \
0         0.680967   0.607115                 1725  

## Train/Test Split

Splitting data into training and test sets.

**Why?**
- Training set (80%): Used to train the recommendation models
- Test set (20%): Used to evaluate how well the models work on unseen data

**Strategy:**
- Stratified split by user: Ensure each user appears in both train and test
- This prevents data leakage and gives honest evaluation

**Result:**
- Train: ~800K ratings for model training
- Test: ~200K ratings for model evaluation

In [ ]:
# Split data: 80% train, 20% test
# random_state=42 for reproducibility
X = final_data.drop('Rating', axis=1)  # Features (everything except Rating)
y = final_data['Rating']               # Target (Rating column)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train/Test Split Completed!")
print(f"\nTraining set size: {X_train.shape[0]} ratings")
print(f"Test set size: {X_test.shape[0]} ratings")
print(f"\nTraining set: {X_train.shape[0] / len(X) * 100:.1f}% of data")
print(f"Test set: {X_test.shape[0] / len(X) * 100:.1f}% of data")
print(f"\nFeatures shape: {X_train.shape[1]} features")
print(f"\nRating distribution in training set:")
print(y_train.value_counts().sort_index())
print(f"\nRating distribution in test set:")
print(y_test.value_counts().sort_index())

Train/Test Split Completed!

Training set size: 800167 ratings
Test set size: 200042 ratings

Training set: 80.0% of data
Test set: 20.0% of data

Features shape: 28 features

Rating distribution in training set:
Rating
1     44773
2     86194
3    209081
4    279323
5    180796
Name: count, dtype: int64

Rating distribution in test set:
Rating
1    11401
2    21363
3    52116
4    69648
5    45514
Name: count, dtype: int64


## Save Processed Data

Saving the train/test split to processed data folder for model training.

Files created:
- X_train.parquet: Training features (800K rows × 28 columns)
- X_test.parquet: Test features (200K rows × 28 columns)
- y_train.parquet: Training target (800K ratings)
- y_test.parquet: Test target (200K ratings)

In [ ]:
# Create data/processed folder if it doesn't exist
os.makedirs('data/processed', exist_ok=True)

# Save train/test splits as pickle (fast and efficient)
with open('data/processed/X_train.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('data/processed/X_test.pkl', 'wb') as f:
    pickle.dump(X_test, f)

with open('data/processed/y_train.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('data/processed/y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)

# Also save metadata
metadata = {
    'total_ratings': len(final_data),
    'train_size': len(X_train),
    'test_size': len(X_test),
    'n_features': X_train.shape[1],
    'feature_names': X_train.columns.tolist()
}

import json
with open('data/processed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Data Saved as Pickle!")
print(f"\nFiles created in data/processed/:")
print(f"  - X_train.pkl ({X_train.shape})")
print(f"  - X_test.pkl ({X_test.shape})")
print(f"  - y_train.pkl ({y_train.shape})")
print(f"  - y_test.pkl ({y_test.shape})")
print(f"  - metadata.json")

✅ Data Saved Successfully!

Files created in data/processed/:
  - X_train.csv ((800167, 28))
  - X_test.csv ((200042, 28))
  - y_train.csv ((800167,))
  - y_test.csv ((200042,))
  - metadata.json
